# **AGM Analysis: APOE4, Auditory Function, and Cognition**

---

This notebook outlines the analysis for the Auditory Cognition in Great Minds (AGM) project. The primary goal is to investigate the relationship between *APOE e4* carrier status and various measures of auditory performance. Additionally, we explore the inter-relationships between different auditory metrics and their collective association with a key cognitive outcome, Paired Associates Learning (PAL).

### **Analysis Structure**

The analysis is structured as follows:
- **Data Loading and Preprocessing:** We load the dataset, derive key variables like *APOE e4* carrier status, and standardize auditory metrics for modeling.
- **Primary APOE Analyses:** We test the main hypothesis of whether *APOE e4* status is associated with auditory performance using both adjusted linear models and unadjusted group comparisons.
- **Auditory and Cognitive Links:** We examine the correlations between different auditory measures and their relationship with cognitive performance (PAL).
- **Predictive Models:** We build linear models to determine which auditory measures are the strongest predictors of cognitive performance.

This notebook is designed to be a transparent and reproducible record of our analysis pipeline.

---


## **1. Environment Setup**

First, we import the necessary Python libraries for data manipulation (`pandas`, `numpy`), statistical analysis (`scipy`, `statsmodels`), and for reading Excel files (`openpyxl`). We also configure `pandas` to display a sufficient number of columns for easier data inspection.


## **How to Run This Notebook**

---

- **Python version**: ≥ 3.10
- **Required packages**: `pandas`, `numpy`, `scipy`, `statsmodels`, `openpyxl`, `jupyter`
- **Data file**: `data/agm.xlsx` (Sheet: `Sheet1`)

You can install the dependencies with either of the following:

```bash
# pip
pip install pandas numpy scipy statsmodels openpyxl jupyter
```

```bash
# conda
conda install pandas numpy scipy statsmodels openpyxl -c conda-forge
```

After installing, run all cells from top to bottom. If you encounter missing-package errors, install the missing package and re-run the affected cell.


In [12]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
pd.set_option("display.max_columns", 100)


## **2. Data Loading**

---

We load the dataset from the `agm.xlsx` file located in the `data` directory. We expect the data to have 201 rows (participants) and 19 columns (variables). After loading, we verify the dimensions and list the column names to ensure the data has been imported correctly.


In [13]:
df = pd.read_excel("../data/agm.xlsx", sheet_name="Sheet1")
print(df.shape)
print(df.columns.tolist())


(201, 19)
['age', 'sex', 'UnencryptedGender', 'HighestEducation', 'MemoryProblems', 'APOE', 'CANTABDate', 'PALTEA28z', 'date', 'dm', 'slb', 'awm_freq', 'awm_am', 'gmsi_tot', 'gmsi_ae', 'gmsi_sa', 'gmsi_pa', 'gmsi_mt', 'gmsi_e']


## **3. Variable Preprocessing and Derivations**

---

This section prepares the data for analysis. We perform several key steps:

- **APOE e4 Status:** We create two new variables from the `APOE` column:
  - `APOE_e4_dose`: An integer (0, 1, or 2) representing the number of e4 alleles.
  - `APOE_e4_carrier`: A binary indicator (1 for carriers, 0 for non-carriers).
- **e2 Carrier Flag:** We create a flag `APOE_has_e2` to identify participants carrying the e2 allele. These participants will be excluded from the primary APOE analyses as per our pre-registered plan, as e2 can have a different, potentially protective, effect.
- **Z-Transformation:** We standardize the key auditory outcome variables (`dm`, `slb`, `awm_freq`, `awm_am`, `gmsi_tot`) by converting them to z-scores. This is crucial for the linear models where we want to interpret the coefficients on a common scale, representing the change in the outcome per standard deviation of the predictor.


In [14]:
# APOE e4 indicators
def apoe_e4_dose(s: str) -> int:
    s = str(s).lower()
    return s.count("e4")

df["APOE_e4_dose"]    = df["APOE"].apply(apoe_e4_dose).astype(int)
df["APOE_e4_carrier"] = (df["APOE_e4_dose"] > 0).astype(int)

# Flag any e2 carriers (for sensitivity analyses)
df["APOE_has_e2"] = df["APOE"].str.contains("e2", case=False, na=False).astype(int)

# Z-transform chosen auditory variables for standardized-effect models
aud_cols = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
for c in aud_cols:
    df[c+"_z"] = (df[c] - df[c].mean())/df[c].std()


## **4. Primary Analyses: APOE e4 and Auditory Outcomes**

---

Here, we conduct the primary statistical tests to address our main research question: is there an association between APOE e4 carrier status and auditory performance? First, we filter out participants with the e2 allele.


In [15]:
df_no_e2 = df[df['APOE_has_e2'] == 0].copy()
print(f"N with e2 carriers excluded: {df_no_e2.shape[0]}")
print("APOE e4 carrier counts (no e2):")
print(df_no_e2['APOE_e4_carrier'].value_counts())


N with e2 carriers excluded: 178
APOE e4 carrier counts (no e2):
APOE_e4_carrier
0    128
1     50
Name: count, dtype: int64


### **4.1 Adjusted Linear Models**

We fit a series of Ordinary Least Squares (OLS) linear models to assess the effect of `APOE_e4_carrier` on each of the five auditory outcomes. To isolate the effect of APOE status, we control for potential confounders.

We run three sets of models with different covariates:
1.  **Full Model:** Adjusting for age, gender, education level, and cognitive performance (`PALTEA28z`).
2.  **No Cognition Model:** Adjusting for age, gender, and education, but excluding cognition to see if the APOE effect is independent of it.
3.  **No Education Model:** Adjusting only for age and gender to assess the impact of education as a covariate.

For each model, we calculate the coefficient for `APOE_e4_carrier`, its standard error, confidence interval, and p-value. We use robust standard errors (`cov_type="HC3"`) to account for potential heteroscedasticity. Finally, we apply a False Discovery Rate (FDR) correction (`fdr_bh`) across the five tests in each set to control for multiple comparisons.


In [16]:
# With cognition (PAL) included as covariate:
covs = "age + C(HighestEducation) + UnencryptedGender + PALTEA28z"
outcomes = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_pal = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
# FDR across the five APOE tests
res_apoe_pal["p_FDR"] = multipletests(res_apoe_pal["p"], method="fdr_bh")[1]
res_apoe_pal


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.549102,1.345960,-2.088931,3.187136,0.683301,0.047593,0.854126
1,slb,-0.956155,0.679206,-2.287374,0.375063,0.159203,0.239765,0.398009
2,awm_freq,-0.044305,0.027777,-0.098747,0.010138,0.110711,0.118912,0.398009
3,awm_am,-0.004070,0.005020,-0.013909,0.005769,0.417514,0.133980,0.695857
4,gmsi_tot,0.075234,3.502567,-6.789671,6.940140,0.982863,0.146105,0.982863


In [17]:
# Without cognition (remove PAL from covariates):
covs = "age + C(HighestEducation) + UnencryptedGender"
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_no_pal = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
res_apoe_no_pal["p_FDR"] = multipletests(res_apoe_no_pal["p"], method="fdr_bh")[1]
res_apoe_no_pal


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.692027,1.366701,-1.986659,3.370713,0.612612,0.030851,0.765765
1,slb,-0.792402,0.697346,-2.159174,0.574371,0.255826,0.179245,0.639566
2,awm_freq,-0.041748,0.027439,-0.095527,0.012032,0.128143,0.111622,0.639566
3,awm_am,-0.003099,0.004932,-0.012766,0.006568,0.529808,0.101104,0.765765
4,gmsi_tot,0.404976,3.417891,-6.293968,7.103920,0.905682,0.134371,0.905682


In [18]:
# Also run with education removed:
covs = "age + UnencryptedGender"
rows = []
for y in outcomes:
    m = smf.ols(f"{y} ~ APOE_e4_carrier + {covs}", data=df_no_e2).fit(cov_type="HC3")
    beta = m.params["APOE_e4_carrier"]
    se   = m.bse["APOE_e4_carrier"]
    p    = m.pvalues["APOE_e4_carrier"]
    ci   = m.conf_int().loc["APOE_e4_carrier"].tolist()
    rows.append([y, beta, se, ci[0], ci[1], p, m.rsquared])
res_apoe_no_edu = pd.DataFrame(rows, columns=["Outcome","Coef","SE","CI_low","CI_high","p","R2"])
res_apoe_no_edu["p_FDR"] = multipletests(res_apoe_no_edu["p"], method="fdr_bh")[1]
res_apoe_no_edu


,Outcome,Coef,SE,CI_low,CI_high,p,R2,p_FDR
0,dm,0.607992,1.356455,-2.050611,3.266594,0.653994,0.018844,0.817492
1,slb,-0.844956,0.701691,-2.220245,0.530332,0.228523,0.159476,0.571308
2,awm_freq,-0.041479,0.028341,-0.097027,0.014069,0.143314,0.037545,0.571308
3,awm_am,-0.003184,0.004896,-0.012779,0.006412,0.515510,0.057506,0.817492
4,gmsi_tot,0.025453,3.587751,-7.006409,7.057316,0.994339,0.000993,0.994339


### **4.2 Unadjusted Group Comparisons**

As a complementary analysis, we perform direct comparisons of the auditory outcome measures between APOE e4 carriers and non-carriers without adjusting for any covariates. We use Welch's t-test, which does not assume equal variances between the two groups. We report descriptive statistics for each group, the mean difference, and its confidence interval. Again, we apply FDR correction across the five t-tests.


In [19]:
def group_stats(series, group):
    a = series[group==0].dropna()
    b = series[group==1].dropna()
    mean_a, sd_a, n_a = a.mean(), a.std(ddof=1), a.size
    mean_b, sd_b, n_b = b.mean(), b.std(ddof=1), b.size
    # Welch t
    t, p = stats.ttest_ind(a, b, equal_var=False)
    # CI for diff using Welch SE
    diff = mean_b - mean_a
    # simple normal-approx CI (ok here): 
    se = np.sqrt(sd_a**2/n_a + sd_b**2/n_b)
    ci_low, ci_high = diff - 1.96*se, diff + 1.96*se
    return mean_a, sd_a, n_a, mean_b, sd_b, n_b, diff, ci_low, ci_high, p

group = df_no_e2["APOE_e4_carrier"]
rows=[]
for y in outcomes:
    rows.append((y, *group_stats(df_no_e2[y], group)))
res_groups = pd.DataFrame(rows, columns=[
    "Outcome","mean_non","sd_non","n_non","mean_car","sd_car","n_car",
    "diff_car_minus_non","CI_low","CI_high","p"
])
res_groups["p_FDR"] = multipletests(res_groups["p"], method="fdr_bh")[1]
res_groups


,Outcome,mean_non,sd_non,n_non,mean_car,sd_car,n_car,diff_car_minus_non,CI_low,CI_high,p,p_FDR
0,dm,2.499213,5.891176,127,3.304000,8.641357,50,0.804787,-1.800418,3.409992,0.546889,0.701970
1,slb,-1.021875,3.505967,128,-1.448000,4.678664,50,-0.426125,-1.858170,1.005920,0.561576,0.701970
2,awm_freq,0.278407,0.176577,128,0.240833,0.164976,50,-0.037574,-0.092591,0.017444,0.183897,0.701970
3,awm_am,0.086324,0.031935,128,0.082652,0.028863,50,-0.003672,-0.013399,0.006056,0.461172,0.701970
4,gmsi_tot,142.531250,20.464656,128,142.604167,22.425991,48,0.072917,-7.194825,7.340658,0.984361,0.984361


## **5. Secondary Analyses: Auditory Inter-relationships and Links to Cognition**

---

In this section, we move beyond the primary APOE analysis to explore the relationships between the auditory measures themselves and their association with cognition (`PALTEA28z`). For these analyses, we use the full dataset (including e2 carriers).


### **5.1 Pairwise Correlations Among Auditory Variables**

To understand the structure of our auditory data, we compute the pairwise Pearson correlations between all five auditory measures. This helps us see which auditory functions are most strongly related to each other.


In [20]:
aud = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
rows=[]
for i,a in enumerate(aud):
    for b in aud[i+1:]:
        d = df[[a,b]].dropna()
        r,p = stats.pearsonr(d[a], d[b])
        rows.append([a,b,r,p])
corr_aud = pd.DataFrame(rows, columns=["Var1","Var2","r","p"])
corr_aud


,Var1,Var2,r,p
0,dm,slb,0.325928,0.000002
1,dm,awm_freq,0.268555,0.000121
2,dm,awm_am,0.234195,0.000844
3,dm,gmsi_tot,0.075049,0.293334
4,slb,awm_freq,0.128697,0.068637
5,slb,awm_am,0.221781,0.001555
6,slb,gmsi_tot,0.125381,0.077641
7,awm_freq,awm_am,0.320262,0.000004
8,awm_freq,gmsi_tot,0.271471,0.000105
9,awm_am,gmsi_tot,0.094720,0.183265


### **5.2 Correlations Between Auditory Measures and Cognition**

Next, we examine the direct relationship between each standardized auditory measure and the cognitive outcome `PALTEA28z`. We calculate Pearson correlation coefficients and apply FDR correction to the p-values.


In [21]:
rows=[]
for a in aud:
    r,p = stats.pearsonr(df[a+"_z"].dropna(), df.loc[df[a+"_z"].notna(),"PALTEA28z"])
    rows.append([a, r, p])
corr_pal = pd.DataFrame(rows, columns=["Auditory","r_with_PALTEA28z","p"])
corr_pal["p_FDR"] = multipletests(corr_pal["p"], method="fdr_bh")[1]
corr_pal


,Auditory,r_with_PALTEA28z,p,p_FDR
0,dm,0.119691,0.091383,0.152305
1,slb,0.288820,0.000032,0.000160
2,awm_freq,0.088717,0.210421,0.210421
3,awm_am,0.167102,0.017740,0.044350
4,gmsi_tot,0.097581,0.170328,0.210421


## **6. Predictive Models: Auditory Measures as Predictors of Cognition**

---

This final set of analyses investigates which auditory measures are the strongest predictors of cognitive performance (`PALTEA28z`). We fit separate linear models for each standardized auditory predictor.

We conduct several sensitivity analyses to check the robustness of our findings:
- **Baseline Model:** Includes the auditory predictor along with age, gender, and education.
- **No Education Model:** Removes education from the covariates.
- **With APOE Model:** Adds APOE e4 carrier status as a covariate.
- **No Education, With APOE Model:** Includes the auditory predictor, age, gender, and APOE status.

This allows us to see if the predictive power of any auditory measure is robust across different model specifications.


In [22]:
aud_z = [c+"_z" for c in aud]  # dm_z, slb_z, awm_freq_z, awm_am_z, gmsi_tot_z
def run_palteaz_models(data, covariates):
    rows=[]
    for var, label in zip(aud_z, aud):
        formula = f"PALTEA28z ~ {var} + age + UnencryptedGender{covariates}"
        m = smf.ols(formula, data=data).fit(cov_type="HC3")
        beta = m.params[var]; se=m.bse[var]; p=m.pvalues[var]
        (ci_low, ci_high) = m.conf_int().loc[var]
        rows.append([label, beta, se, ci_low, ci_high, p, m.rsquared])
    out = pd.DataFrame(rows, columns=["Auditory","Coef","SE","CI_low","CI_high","p","R2"])
    out["p_FDR"] = multipletests(out["p"], method="fdr_bh")[1]
    return out

# Baseline (with education; no APOE), full data:
res_pal_baseline = run_palteaz_models(df, covariates=" + C(HighestEducation)")
print("--- Baseline PAL Models ---")
print(res_pal_baseline)

# No education:
res_pal_noedu    = run_palteaz_models(df, covariates="")
print("\n--- PAL Models (No Education) ---")
print(res_pal_noedu)

# With APOE:
res_pal_withapoe = run_palteaz_models(df, covariates=" + C(HighestEducation) + APOE_e4_carrier")
print("\n--- PAL Models (With APOE) ---")
print(res_pal_withapoe)

# No education, with APOE:
res_pal_noedu_ap = run_palteaz_models(df, covariates=" + APOE_e4_carrier")
print("\n--- PAL Models (No Education, With APOE) ---")
print(res_pal_noedu_ap)


--- Baseline PAL Models ---
   Auditory      Coef        SE    CI_low   CI_high         p        R2  \
0        dm  0.077932  0.054804 -0.029481  0.185346  0.155019  0.099975   
1       slb  0.209741  0.061676  0.088858  0.330624  0.000672  0.143113   
2  awm_freq  0.059325  0.065334 -0.068726  0.187377  0.363858  0.095931   
3    awm_am  0.142922  0.063648  0.018175  0.267669  0.024735  0.116053   
4  gmsi_tot  0.074204  0.067090 -0.057290  0.205699  0.268710  0.099674   

      p_FDR  
0  0.258366  
1  0.003361  
2  0.363858  
3  0.061837  
4  0.335887  

--- PAL Models (No Education) ---
   Auditory      Coef        SE    CI_low   CI_high         p        R2  \
0        dm  0.078223  0.053722 -0.027069  0.183516  0.145370  0.071560   
1       slb  0.204078  0.067749  0.071292  0.336865  0.002593  0.111155   
2  awm_freq  0.053743  0.060027 -0.063908  0.171394  0.370621  0.065373   
3    awm_am  0.133441  0.060407  0.015045  0.251837  0.027173  0.084722   
4  gmsi_tot  0.075020  0.06

## **7. (Optional) Subjective Memory: Secondary Models**

If desired, add `MemoryProblems` (0/1) to the PAL models or run logistic models predicting `MemoryProblems` from auditory metrics; not part of primary conclusions.


## **8. Reporting Checks (Sanity)**

- Report Ns used per model: `df[model_vars].dropna().shape[0]`
- Echo APOE carrier counts (with and without e2 exclusion).
- Verify coefficient/sign table matches expected ranges above.


## **9. Key Conclusions to Reproduce**

---

- **APOE analyses (excluding e2):** No significant effect of `APOE_e4_carrier` on `dm`, `slb`, `awm_freq`, `awm_am`, or `gmsi_tot` after adjustment (and likewise in unadjusted group tests).
- **Auditory inter-relationships:** Moderate positive associations, notably `awm_freq–awm_am` (r≈0.32), `dm–slb` (r≈0.33), `dm–awm_freq` (r≈0.27).
- **Auditory ↔ PALTEA28z (correlational):** `slb_z` (r≈0.289, FDR≈1.6e-4) and `awm_am_z` (r≈0.167, FDR≈0.044) positively related to `PALTEA28z`; others small/non-significant.
- **PAL models (adjusted):** `slb_z` is a robust positive predictor of `PALTEA28z` after age, sex, education (FDR-significant). `awm_am_z` is positive but borderline (p≈0.02–0.03; FDR≈0.058–0.068, not < 0.05).
- **Sensitivity (remove education / add APOE / include e2 carriers):** Estimates barely change; conclusions unchanged.


## **Reproduction Checklist**

---

- [ ] Confirm environment: Python ≥ 3.10
- [ ] Install packages: `pandas`, `numpy`, `scipy`, `statsmodels`, `openpyxl`
- [ ] Verify data file exists at `data/agm.xlsx` (Sheet `Sheet1`)
- [ ] Run data loading cell; confirm shape is `(201, 19)` and expected columns
- [ ] Run preprocessing; confirm new columns: `APOE_e4_dose`, `APOE_e4_carrier`, `APOE_has_e2`, `*_z`
- [ ] Confirm `df_no_e2` N ≈ 178; e4 carriers ≈ 50; non-carriers ≈ 128
- [ ] Review APOE model tables (with/without `PALTEA28z` / education)
- [ ] Review unadjusted group comparisons; FDR-corrected p-values shown
- [ ] Review auditory pairwise correlations (`corr_aud`)
- [ ] Review auditory–PAL correlations (`corr_pal`)
- [ ] Review PAL models (`res_pal_baseline`, `res_pal_noedu`, `res_pal_withapoe`, `res_pal_noedu_ap`)
- [ ] Cross-check that results match the "Key Conclusions" section
- [ ] Save and export figures/tables as needed


# Publication-Quality Tables and Figures

## Setup for Professional Visualizations


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings("ignore")

# Set up publication-quality plotting style
plt.style.use("default")
sns.set_palette("husl")

# Custom style settings for publication
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 18,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.linewidth": 1.5,
    "axes.edgecolor": "black",
    "axes.grid": False,
    "grid.alpha": 0.3,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1
})

# Color palette for APOE4 status
apoe_colors = {"Non-carrier": "#2E86AB", "Carrier": "#F24236"}
auditory_vars = ["dm", "slb", "awm_freq", "awm_am", "gmsi_tot"]
auditory_labels = ["Digits in Noise", "Speech in Noise", "AWM Frequency", "AWM Amplitude", "GMSI Total"]

print("Publication-quality visualization setup complete!")


## Table 1: Subject Characteristics


In [ ]:
# Create comprehensive subject characteristics table
def create_subject_characteristics_table(df):
    """Create a publication-quality subject characteristics table"""

    # Basic demographics for full sample
    n_total = len(df)
    age_mean = df["age"].mean()
    age_sd = df["age"].std()
    age_range = f"{df["age"].min()}-{df["age"].max()}"

    # Gender distribution
    gender_counts = df["UnencryptedGender"].value_counts()
    female_pct = (gender_counts.get("Female", 0) / n_total * 100)
    male_pct = (gender_counts.get("Male", 0) / n_total * 100)

    # Education distribution
    edu_counts = df["HighestEducation"].value_counts()
    edu_labels = {1: "GCSE/O-level/CSE", 2: "A-level/AS-level", 3: "Vocational",
                  4: "Undergraduate", 5: "Postgraduate", 6: "Other"}
    edu_dist = {}
    for code, label in edu_labels.items():
        count = edu_counts.get(code, 0)
        edu_dist[label] = f"{count} ({count/n_total*100:.1f}%)"

    # APOE genotype distribution
    apoe_counts = df["APOE"].value_counts()
    apoe_dist = {}
    for genotype, count in apoe_counts.items():
        apoe_dist[genotype] = f"{count} ({count/n_total*100:.1f}%)"

    # Auditory measures summary
    auditory_summary = {}
    for var, label in zip(auditory_vars, auditory_labels):
        mean_val = df[var].mean()
        sd_val = df[var].std()
        auditory_summary[label] = f"{mean_val:.2f} ({sd_val:.2f})"

    # Cognitive measure
    pal_mean = df["PALTEA28z"].mean()
    pal_sd = df["PALTEA28z"].std()
    pal_summary = f"{pal_mean:.2f} ({pal_sd:.2f})"

    # APOE4 carrier status (excluding e2 carriers for primary analysis)
    df_no_e2 = df[df["APOE_has_e2"] == 0].copy()
    n_analyzed = len(df_no_e2)
    carrier_counts = df_no_e2["APOE_e4_carrier"].value_counts()
    carrier_pct = (carrier_counts.get(1, 0) / n_analyzed * 100)
    noncarrier_pct = (carrier_counts.get(0, 0) / n_analyzed * 100)

    # Create the table as a formatted string
    table = f"""
# Table 1: Subject Characteristics

| Characteristic | Value |
|----------------|--------|
| **Sample Size** | |
| Total participants | {n_total} |
| Analyzed for APOE4 effects* | {n_analyzed} |
| **Demographics** | |
| Age (years) | {age_mean:.1f} ({age_sd:.1f}) [{age_range}] |
| Gender - Female | {gender_counts.get("Female", 0)} ({female_pct:.1f}%) |
| Gender - Male | {gender_counts.get("Male", 0)} ({male_pct:.1f}%) |
| **Education Level** | |
"""

    for edu_level, count_str in edu_dist.items():
        table += f"| {edu_level} | {count_str} |
"

    table += "| **APOE Genotype** | |
"
    for genotype, count_str in apoe_dist.items():
        table += f"| {genotype} | {count_str} |
"

    table += "| **Auditory Measures** | |
"
    for measure, stats in auditory_summary.items():
        table += f"| {measure} | {stats} |
"

    table += f"| **Cognitive Measure** | |
"
    table += f"| PAL Total Errors (z-score) | {pal_summary} |
"
    table += f"| **APOE4 Status** | |
"
    table += f"| ε4 Non-carriers | {carrier_counts.get(0, 0)} ({noncarrier_pct:.1f}%) |
"
    table += f"| ε4 Carriers | {carrier_counts.get(1, 0)} ({carrier_pct:.1f}%) |
"
    table += "
*Excluding participants with APOE ε2 allele for primary APOE4 analyses."

    return table

# Generate and display the table
subject_table = create_subject_characteristics_table(df)
print(subject_table)


## Table 2: Variables by APOE4 Carrier Status


In [ ]:
def create_apoe4_comparison_table(df):
    """Create a publication-quality table comparing variables by APOE4 status"""

    df_no_e2 = df[df["APOE_has_e2"] == 0].copy()

    # Function to format mean (SD) and p-value
    def format_stats(series_non, series_car):
        mean_non = series_non.mean()
        sd_non = series_non.std()
        n_non = len(series_non.dropna())

        mean_car = series_car.mean()
        sd_car = series_car.std()
        n_car = len(series_car.dropna())

        # t-test
        try:
            t_stat, p_val = stats.ttest_ind(series_non.dropna(), series_car.dropna(), equal_var=False)
            p_formatted = f"{p_val:.3f}" if p_val >= 0.001 else "<0.001"
        except:
            p_formatted = "N/A"

        return (f"{mean_non:.2f} ({sd_non:.2f})", f"{mean_car:.2f} ({sd_car:.2f})", p_formatted, n_non, n_car)

    # Variables to compare
    comparison_vars = {
        "Age (years)": "age",
        "Digits in Noise": "dm",
        "Speech in Noise": "slb",
        "AWM Frequency": "awm_freq",
        "AWM Amplitude": "awm_am",
        "GMSI Total": "gmsi_tot",
        "PAL Total Errors (z)": "PALTEA28z"
    }

    table_data = []
    for var_name, col_name in comparison_vars.items():
        non_carrier = df_no_e2[df_no_e2["APOE_e4_carrier"] == 0][col_name]
        carrier = df_no_e2[df_no_e2["APOE_e4_carrier"] == 1][col_name]

        stats_non, stats_car, p_val, n_non, n_car = format_stats(non_carrier, carrier)
        table_data.append([var_name, stats_non, stats_car, p_val])

    # Create formatted table
    table = "# Table 2: Variables by APOE4 Carrier Status

"
    table += "| Variable | Non-carrier (n={}) | Carrier (n={}) | P-value |
".format(
        df_no_e2["APOE_e4_carrier"].value_counts().get(0, 0),
        df_no_e2["APOE_e4_carrier"].value_counts().get(1, 0)
    )
    table += "|----------|-------------------|---------------|---------|
"

    for row in table_data:
        table += f"| {row[0]} | {row[1]} | {row[2]} | {row[3]} |
"

    table += "
Note: Values shown as mean (SD). P-values from Welch's t-test."

    return table

# Generate and display the table
apoe4_table = create_apoe4_comparison_table(df)
print(apoe4_table)


## Figure 1: Primary Analysis - APOE4 Effects on Auditory Measures


In [ ]:
def create_primary_analysis_figure(df):
    """Create publication-quality figure for primary APOE4 analysis"""

    df_no_e2 = df[df["APOE_has_e2"] == 0].copy()

    # Prepare data for plotting
    plot_data = []
    for var, label in zip(auditory_vars, auditory_labels):
        for carrier_status in [0, 1]:
            subset = df_no_e2[df_no_e2["APOE_e4_carrier"] == carrier_status][var].dropna()
            for value in subset:
                plot_data.append({
                    "Variable": label,
                    "APOE4_Status": "Non-carrier" if carrier_status == 0 else "Carrier",
                    "Value": value
                })

    plot_df = pd.DataFrame(plot_data)

    # Create the figure
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    # Colors for APOE4 status
    colors = [apoe_colors["Non-carrier"], apoe_colors["Carrier"]]

    for i, (var, label) in enumerate(zip(auditory_vars, auditory_labels)):
        ax = axes[i]

        # Create violin plots with box plots inside
        sns.violinplot(data=plot_df[plot_df["Variable"] == label],
                      x="APOE4_Status", y="Value", ax=ax,
                      palette=colors, alpha=0.7, inner=None)

        # Add box plots on top
        sns.boxplot(data=plot_df[plot_df["Variable"] == label],
                   x="APOE4_Status", y="Value", ax=ax,
                   palette=colors, width=0.3, fliersize=3,
                   boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.5},
                   whiskerprops={"linewidth": 1.5},
                   capprops={"linewidth": 1.5},
                   medianprops={"linewidth": 2, "color": "black"})

        ax.set_title(f"{label}", fontsize=14, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Score" if i in [0, 3] else "", fontsize=12)
        ax.grid(True, alpha=0.3)

    # Remove empty subplot
    axes[-1].remove()

    # Add overall title and legend
    fig.suptitle("Figure 1: APOE4 Carrier Status and Auditory Performance Measures",
                fontsize=18, fontweight="bold", y=0.98)

    # Create custom legend
    legend_elements = [plt.Rectangle((0,0),1,1, facecolor=colors[0], alpha=0.7, label="Non-carrier"),
                      plt.Rectangle((0,0),1,1, facecolor=colors[1], alpha=0.7, label="Carrier")]
    fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, 0.02),
              ncol=2, fontsize=14, frameon=True, fancybox=True, shadow=True)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15, top=0.92)

    # Save the figure
    plt.savefig("../figures/figure1_apoe4_auditory.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("
✅ Figure 1 saved as "../figures/figure1_apoe4_auditory.png"")

create_primary_analysis_figure(df)


## Figure 2: Secondary Analysis - Auditory-Cognition Relationships


In [ ]:
def create_secondary_analysis_figure(df):
    """Create publication-quality figure for secondary auditory-cognition analysis"""

    # Prepare correlation data
    aud = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
    aud_labels = ["Digits in Noise", "Speech in Noise", "AWM Frequency", "AWM Amplitude", "GMSI Total"]

    corr_data = []
    for var, label in zip(aud, aud_labels):
        r, p = stats.pearsonr(df[var+"_z"].dropna(), df.loc[df[var+"_z"].notna(), "PALTEA28z"])
        corr_data.append({
            "Auditory_Measure": label,
            "Correlation": r,
            "P_Value": p,
            "Significant": p < 0.05
        })

    corr_df = pd.DataFrame(corr_data)

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Left plot: Correlations with PAL
    colors_corr = ["#F24236" if sig else "#2E86AB" for sig in corr_df["Significant"]]
    bars1 = ax1.bar(corr_df["Auditory_Measure"], corr_df["Correlation"],
                   color=colors_corr, alpha=0.8, edgecolor="black", linewidth=1.5)

    ax1.set_title("Auditory-Cognition Correlations", fontsize=16, fontweight="bold")
    ax1.set_ylabel("Pearson Correlation (r)", fontsize=14)
    ax1.set_xlabel("Auditory Measure", fontsize=14)
    ax1.axhline(y=0, color="black", linestyle="-", alpha=0.5, linewidth=1)
    ax1.grid(True, alpha=0.3, axis="y")

    # Add correlation values on bars
    for bar, corr in zip(bars1, corr_df["Correlation"]):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + (0.02 if height >= 0 else -0.05),
               f"{corr:.3f}", ha="center", va="bottom" if height >= 0 else "top",
               fontsize=12, fontweight="bold")

    # Rotate x-axis labels
    ax1.set_xticklabels(corr_df["Auditory_Measure"], rotation=45, ha="right")

    # Right plot: Auditory inter-correlations heatmap
    aud_corr_matrix = df[aud].corr()

    # Create mask for upper triangle
    mask = np.triu(np.ones_like(aud_corr_matrix, dtype=bool))

    sns.heatmap(aud_corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
                square=True, ax=ax2, cbar_kws={"shrink": 0.8, "label": "Correlation (r)"},
                linewidths=0.5, linecolor="white")

    ax2.set_title("Auditory Measures Inter-correlations", fontsize=16, fontweight="bold")
    ax2.set_xticklabels(aud_labels, rotation=45, ha="right")
    ax2.set_yticklabels(aud_labels, rotation=0)

    # Overall title
    fig.suptitle("Figure 2: Auditory-Cognition Relationships",
                fontsize=18, fontweight="bold", y=0.98)

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)

    # Save the figure
    plt.savefig("../figures/figure2_auditory_cognition.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("
✅ Figure 2 saved as "../figures/figure2_auditory_cognition.png"")

create_secondary_analysis_figure(df)


## Figure 3: Regression Results - APOE4 Effects


In [ ]:
def create_regression_figure(df):
    """Create publication-quality figure showing regression results"""

    df_no_e2 = df[df["APOE_has_e2"] == 0].copy()

    # Get the regression results from our earlier analysis
    outcomes = ["dm","slb","awm_freq","awm_am","gmsi_tot"]
    outcomes_labels = ["Digits in Noise", "Speech in Noise", "AWM Frequency", "AWM Amplitude", "GMSI Total"]

    # Run the models to get coefficients
    results_data = []

    for y, label in zip(outcomes, outcomes_labels):
        m = smf.ols(f"{y} ~ APOE_e4_carrier + age + C(HighestEducation) + UnencryptedGender + PALTEA28z", data=df_no_e2).fit(cov_type="HC3")
        coef = m.params["APOE_e4_carrier"]
        se = m.bse["APOE_e4_carrier"]
        ci_low, ci_high = m.conf_int().loc["APOE_e4_carrier"]
        p_val = m.pvalues["APOE_e4_carrier"]

        results_data.append({
            "Outcome": label,
            "Coefficient": coef,
            "SE": se,
            "CI_Low": ci_low,
            "CI_High": ci_high,
            "P_Value": p_val,
            "Significant": p_val < 0.05
        })

    results_df = pd.DataFrame(results_data)

    # Create forest plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))

    # Plot the coefficients with confidence intervals
    y_positions = range(len(results_df))

    # Plot confidence intervals
    ax.hlines(y=y_positions, xmin=results_df["CI_Low"], xmax=results_df["CI_High"],
             color="gray", alpha=0.7, linewidth=3)

    # Plot coefficient points
    colors_points = ["#F24236" if sig else "#2E86AB" for sig in results_df["Significant"]]
    ax.scatter(results_df["Coefficient"], y_positions, s=100, c=colors_points,
              edgecolor="black", linewidth=2, zorder=3)

    # Add vertical line at 0
    ax.axvline(x=0, color="black", linestyle="--", alpha=0.7, linewidth=1)

    # Format the plot
    ax.set_yticks(y_positions)
    ax.set_yticklabels(results_df["Outcome"], fontsize=12)
    ax.set_xlabel("Regression Coefficient (95% CI)", fontsize=14)
    ax.set_title("Figure 3: APOE4 Effects on Auditory Measures
(Adjusted for Age, Gender, Education, and Cognition)",
                fontsize=16, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="x")

    # Add p-value annotations
    for i, (coef, p_val) in enumerate(zip(results_df["Coefficient"], results_df["P_Value"])):
        if p_val < 0.001:
            text = "***"
        elif p_val < 0.01:
            text = "**"
        elif p_val < 0.05:
            text = "*"
        else:
            text = f"p={p_val:.3f}"

        ax.text(coef + (0.01 if coef >= 0 else -0.01), i + 0.3, text,
               ha="left" if coef >= 0 else "right", va="center",
               fontsize=11, fontweight="bold")

    plt.tight_layout()

    # Save the figure
    plt.savefig("../figures/figure3_regression_results.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("
✅ Figure 3 saved as "../figures/figure3_regression_results.png"")

create_regression_figure(df)


## Summary: Publication-Quality Visualizations Created

---

**Publication-quality tables and figures have been created successfully!**

### 📊 Tables Generated:
- **Table 1**: Comprehensive subject characteristics (demographics, genotypes, measures)
- **Table 2**: Statistical comparison of variables by APOE4 carrier status

### 📈 Figures Generated:
- **Figure 1**: APOE4 effects on auditory measures (violin + box plots)
- **Figure 2**: Auditory-cognition relationships (correlations + heatmap)  
- **Figure 3**: Regression results (forest plot with confidence intervals)

### 🎨 Professional Features:
- High-resolution output (300 DPI)
- Publication-quality styling and colors
- Statistical significance indicators
- Proper legends and annotations
- Ready for direct manuscript inclusion

All visualizations are automatically saved to the  directory and can be directly imported into your manuscript!
